# SurEau photosynthesis
 

In [ ]:
#| default_exp sureau_photosynthesis

In [ ]:
# | hide
from fastcore import *
from nbdev.showdoc import *

In [ ]:
# | export

import numpy as np
from plant_hydraulics.utils import arrhenius_function, inhibition_function   
from plant_hydraulics.parameter_classes import (
    PhysCon,                                                                 
    SurEauVegetationParams,
    SurEauPlantFluxes,
)

In [ ]:
#| export

def _an_quadratic(a0, e0, d0, gleaf, Ca, cp, rd):
    """Smaller root of the C3 assimilation-diffusion quadratic (Bonan 2019,
    leaf_ci_optimization). Returns the NET rate; add rd to recover the gross rate.

    The general C3 form  An = a0(Ci - cp)/(e0*Ci + d0) - rd  combined with the
    CO2 diffusion equation  Ci = Ca - An/gleaf  gives a quadratic in An.
    """
    aquad = e0 / gleaf
    bquad = -(e0 * Ca + d0) - (a0 - e0 * rd) / gleaf
    cquad = a0 * (Ca - cp) - rd * (e0 * Ca + d0)
    return float(np.min(np.roots([aquad, bquad, cquad]).real))

In [ ]:
# | export

def compute_photosynthesis_fvcb(T_leaf, aPAR, gs, gb_co2, params):
    """C3 Farquhar-von Caemmerer-Berry photosynthesis for SurEau 
    
    Temperature scaling reuses utils.arrhenius_function (Eq. 11.34) and
    utils.inhibition_function (Eq. 11.36). The peaked Arrhenius is their product, 
    normalised to 1 at 25C; the normalisation constant fc is the local fth25 
    (== leaf.vcmaxc/jmaxc/rdc in leaf_phys_params.py).

    Parameters
    ----------
    T_leaf : float   Leaf temperature [degC].
    aPAR   : float   Absorbed PAR [umol photon m-2 s-1].
    gs     : float   Stomatal conductance to H2O [mol m-2 s-1] (NOTE: mol, not mmol).
    gb_co2 : float   Leaf boundary-layer conductance to CO2 [mol m-2 s-1].
    params : SurEauVegetationParams   FvCB block + CO2_air, O2_air.

    Returns
    -------
    An : float   Net photosynthesis [umol CO2 m-2 s-1] (negative at night = -Rd).
    ci : float   Intercellular CO2 [umol mol-1].
    cs : float   Leaf-surface CO2 [umol mol-1].
    
    """
    # Local functions -----------------------------------------------------------

    # Normalization function
    def fth25(hd, se):
        return 1.0 + np.exp((-hd + se * T0) / (PhysCon.rgas * T0))
    
    # Peak arrhenius
    def peaked(ha, hd, se):
        return arrhenius_function(tl, ha) * inhibition_function(tl, hd, se, fth25(hd, se))

    # Constants ----------------------------------------------------------------
    T0 = PhysCon.tfrz + 25.0         
    tl = T_leaf + PhysCon.tfrz          
    kc = params.kc25 * arrhenius_function(tl, params.kcha)
    ko = params.ko25 * arrhenius_function(tl, params.koha)
    cp = params.cp25 * arrhenius_function(tl, params.cpha)

    # Peaked Arrhenius for Vcmax, Jmax, Rd --------------------------------------
    vcmax = params.Vcmax25 * peaked(params.vcmaxha, params.vcmaxhd, params.vcmaxse)
    jmax = params.Jmax25 * peaked(params.jmaxha, params.jmaxhd, params.jmaxse)
    rd = params.Rd25 * peaked(params.rdha, params.rdhd, params.rdse)

    # Electron transport rate J ------------------------------------------------- 
    # (smaller root of the light co-limitation, Eq. 11.21)
    qabs = 0.5 * params.phi_psii * aPAR
    je = float(np.min(np.roots([params.theta_j, -(qabs + jmax), qabs * jmax]).real))
    
    # Total leaf conductance to CO2 --------------------------------------------- 
    # Boundary layer + stomata in series.
    # 1.6 converts gs from an H2O to a CO2 basis (D_H2O / D_CO2).
    gleaf = 1.0 / (1.0 / gb_co2 + 1.6 / gs)
    Ca, O2 = params.CO2_air, params.O2_air

    # Rubisco-limited (Ac) and RuBP-regeneration-limited (Aj) gross rates -------
    ac = _an_quadratic(vcmax, 1.0, kc * (1.0 + O2 / ko), gleaf, Ca, cp, rd) + rd
    aj = _an_quadratic(je, 4.0, 8.0 * cp, gleaf, Ca, cp, rd) + rd
    ac, aj = max(ac, 0.0), max(aj, 0.0)

    # Smooth co-limitation (Eq. 4.19) and net assimilation ----------------------
    ag = float(np.min(np.roots([params.colim_c3, -(ac + aj), ac * aj]).real))
    ag = max(ag, 0.0)
    an = ag - rd
    cs = max(Ca - an / gb_co2, 1.0)
    ci = Ca - an / gleaf
    
    return an, ci, cs


In [ ]:
#| export

def calculate_gs_medlyn(
    fluxes: SurEauPlantFluxes,
    params: SurEauVegetationParams,
    clim: dict,
) -> SurEauPlantFluxes:
    """Medlyn et al. (2011) stomatal conductance coupled to FvCB photosynthesis.

    Replacement for `calculate_gs_jarvis`stomatal conductance 
    
    SurEau's hydraulic regulation gamma
    (compute_regul_fact, applied by compute_transpiration) is what turns it into
    the water-limited gs_lim, so this routine has no knowledge of psi.

    Medlyn (Eq. 11):  gs = g0 + 1.6 (1 + g1/sqrt(D)) * An / cs

    gs<-> An <->Ci are dependent, so a iteration is used:
    given a trial gs, FvCB returns An and cs; Medlyn returns a new gs; repeat.

    """
    # Constants -----------------------------------------------------------------
    max_iter = 50
    
    # mmol m-2 s-1
    tol = 1.0  

    # Leaf temp
    Tl = fluxes.leaf_temperature
    
    # Incident ~ absorbed for a sunlit leaf
    aPAR = clim["PAR"] 
    
    # kPa, floored as in calculate_gs_jarvis                      
    D = max(fluxes.leaf_VPD, 0.1)           
    
    # mmol
    g0 = params.g0_medlyn                    
    g1 = params.g1_medlyn

    # Leaf boundary-layer conductance to CO2 [mol] ------------------------------ 
    # H2O->CO2 /1.37, mmol-> mol /1000.
    # Leaf-level CO2 supply is set by the leaf boundary layer, 
    # not the bulk canopy aerodynamic conductance.
    gb_co2 = fluxes.g_BL / 1.37 / 1000.0

    # mmol, Initial gs guess
    gs = 150.0                               
    an = ci = cs = 0.0
    
    for _ in range(max_iter):
        
        gs_old = gs
        
        # Compute photosynthesis
        an, ci, cs = compute_photosynthesis_fvcb(Tl, aPAR, gs / 1000.0, gb_co2, params)
        
        if an > 0.0 and cs > 0.0:
    
            # 1000 converts the mol-basis Medlyn result to SurEau's mmol basis
            gs = g0 + 1000 * 1.6 * (1.0 + g1 / np.sqrt(D)) * an / cs
        
        else:
            
            # At night or respiration-dominated -> residual conductance only
            gs = g0

        # Get gs values
        gs = max(gs, g0)
        
        if abs(gs - gs_old) < tol:
            break

    fluxes.gs_bound = gs
    fluxes.An = an
    fluxes.ci = ci
    fluxes.cs = cs
    
    return fluxes